In [13]:
import numpy as np

In [14]:
def c_to_f(Temp_C):
    return Temp_C * (9/5) + 32

In [15]:
# The goal will be to minimize the absolute value of this
# In the future: Add solar heat gain and internal heat gain (from stoves, people, etc. inside the building)
def building_heat_transfer(T_internal_F, T_external_F, R_wall, area, ACH, volume):
    """
    Calculates total heat loss rate from building per second
    in Watts, given internal and external temperature

    T_internal_F: Temperature inside the room in Fahrenheit
    T_external_F: Temperature outside the building in Fahrenheit
    R_wall: insulation R value of wall
    ACH: Air Changes per Hour by infiltration
    volume:
    area: total surface area through which heat can escape (walls, ceiling, floor, windows)
    """
    T_internal_K = (T_internal_F - 32) * 5/9 + 273.15
    T_external_K = (T_external_F - 32) * 5/9 + 273.15

    delta_T = T_internal_K - T_external_K
    # conductive heat loss through building
    Q_conductive = (delta_T / R_wall) * area

    # infiltration loss - air leaking in and out
    rho_air = 1.2 # unit - kg/m3
    cp_air = 1005 # unit J/kg.K
    Q_infiltration = (ACH / 3600) * volume * rho_air * cp_air * delta_T

    Q_net = Q_conductive + Q_infiltration

    #print("Q conductive: ", Q_conductive)
    #print("Q infiltration: ", Q_infiltration)
    return Q_net
    # return Q_conductive

In [16]:
def sat_pressure(T_F):
    """
    Calculates saturation pressure, the maximum pressure 
    water vapor can exert at a given temperature
    
    T_F: Temperature in Fahrenheit
    """
    # Convert to Celsius for ASHRAE Eq. 5 (IAPWS-IF97)
    # ASHRAE Fundamentals 2025, Chapter 1, Equation 5
    T_C = (T_F - 32) * 5/9
    T_K = T_C + 273.15
    theta = T_K + (-0.238555575678e0) / (T_K - 0.650175348448e3)
    A = theta**2 + 0.116705214528e4 * theta - 0.724213167032e6
    B = -0.170738469401e2 * theta**2 + 0.120208247025e5 * theta - 0.323255503223e7
    C = 0.149151086135e2 * theta**2 - 0.482326573616e4 * theta + 0.405113405421e6
    p_ws = 1000 * (2*C / (-B + (B**2 - 4*A*C)**0.5))**4  # kPa
    return p_ws

In [17]:
def indoor_humidity(T_internal_F, T_external_F, RH_external, cooling, cooling_rh_cap=0.60):
    """
    Calculates indoor relative humidity by assuming indoor air 
    carries the same absolute moisture content as outdoor air, 
    then recalculates what that moisture level feels like at 
    the warmer indoor temperature

    T_internal_F: Indoor temperature in Fahrenheit
    T_external_F: External Temperature in Fahrenheit
    RH_external: External Relative Humidity
    """
    """
    ASHRAE Fundamentals 2025 Chapter 1
    Humidity ratio W: Equation 21
    Saturation pressure: Equation 5
    """
    P_atm = 101.325  # kPa

    # Outdoor absolute humidity (humidity ratio W)
    P_sat_ext = sat_pressure(T_external_F)
    p_w = RH_external * P_sat_ext
    W = 0.621945 * p_w / (P_atm - p_w)  # Eq. 21 ASHRAE Ch.1

    # Indoor RH at indoor temp, same W (infiltration assumption)
    P_sat_int = sat_pressure(T_internal_F)
    p_w_int = W * P_atm / (0.621945 + W)
    RH_internal = p_w_int / P_sat_int
    RH_internal = min(RH_internal, 1.0)
    
    # simple approximation: AC dehumidifies in cooling mode
    if cooling:
        RH_internal = min(RH_internal, cooling_rh_cap)

    return RH_internal

In [18]:
def wall_temp(T_internal_F, Q_net, area, R_film=0.13):
    """
    Calculates inner wall temperature at steady state

    T_internal_F: Temperature inside the room in Fahrenheit
    R_film: insulation R value of the layer of air next to the inner faces of the wall
    area: total surface area through which heat can escape (walls, ceiling, floor, windows)
    """
    T_internal_K = (T_internal_F - 32) * 5/9 + 273.15    
    T_wall_K = T_internal_K - (Q_net * R_film / area) 
    T_wall_F = (T_wall_K - 273.15) * 9/5 + 32  
    return T_wall_F

In [19]:
def comfort_model(T_internal_F, RH_internal, T_wall_F, clothing_factor=0.5):
    """
    Calculates total heat loss from the human body via radiation to walls, 
    convection to air, and evaporation. Compares against metabolic rate to 
    give a Hot/Cold/Good verdict.

    T_internal_F: Indoor Temperature in Fahrenheit
    RH_internal: Indoor Relative Humidity
    T_wall_F: Wall Temperature in Fahrenheit
    """
    
    """
    Radiation + convection heat loss from human body
    Taken from Dr. Gray's matlab code
    ASHRAE Fundamentals 2025 Chapter 9
    """
    
    epsilon = 0.98
    sigma = 5.6703e-8
    H = 3.5 # Lowered convection coefficient from 5 to 3.5
    T_skin_K = 306.15 # 33 C

    T_wall_K = (T_wall_F - 32) * 5/9 + 273.15
    T_air_K  = (T_internal_F - 32) * 5/9 + 273.15

    q_rad  = epsilon * sigma * (T_skin_K**4 - T_wall_K**4)
    q_conv = H * (T_skin_K - T_air_K)

    q_rad *= clothing_factor
    q_conv *= clothing_factor
    
    # Evaporative heat loss — ASHRAE Fundamentals Ch.9
    # q_evap decreases as RH increases (less evaporation possible)
    # At rest: ~10 W/m2 at low RH, approaches 0 at high RH
    # Tweaked to 5
    q_evap = 5 * (1 - RH_internal)  # simplified linear approximation
    Q_net = q_rad + q_conv + q_evap

    METABOLIC_RATE = 60  # W/m2, seated quiet, ASHRAE Fundamentals Ch.9 Table 4

    tolerance = 12  # W/m2 either side — needs calibration from your survey data

    if Q_net < METABOLIC_RATE - tolerance:
        verdict = "Hot"
    elif Q_net > METABOLIC_RATE + tolerance:
        verdict = "Cold"
    else:
        verdict = "Good"

    return Q_net, verdict




In [20]:
def steady_state_model(T_setpoint_F, T_external_F, RH_external, R_wall=2.3, A_envelope=70, ACH=0.5, volume=30, print_output=True):
    """
    Takes all 3 steps together
    """
    Q_net = building_heat_transfer(T_setpoint_F, T_external_F, R_wall, A_envelope, ACH, volume)
    if Q_net > 0:
        mode = "heating"
    elif Q_net < 0:
        mode = "cooling"
    else:
        mode = "neutral"

    RH_internal = indoor_humidity(T_setpoint_F, T_external_F, RH_external, cooling=(mode == "cooling"))
    
    T_wall_F = wall_temp(T_setpoint_F, Q_net, A_envelope)
    user_Q_net, verdict = comfort_model(T_setpoint_F, RH_internal, T_wall_F)
    if (print_output):
        print("Mode:", mode)
        print("Internal RH: ", RH_internal)
        print("Building heat loss: ", Q_net)
        print("Wall temperature (F): ", T_wall_F)
        print()
        print("Heat loss of user (watts): ", user_Q_net)
        print("Verdict: ", verdict)
        print()
    return mode, user_Q_net, verdict, RH_internal, Q_net, T_wall_F

# Sweep outdoor conditions to build database

## Reference values for:
### ACH (Air Changes per Hour by infiltration): 0.5
### R_wall: 2.3 $m^2 K/W$
### R_film: 0.13 $m^2 K/W$
### Volume: 30 $m^3$ for a room (500 for whole home)
### Area: 70 $m^2$ for a room (500 for whole home)
We might make adjustments to these values later

In [22]:
# December 2025, row 28 (12/1/2025  9:35:00 AM)
s = steady_state_model(68, c_to_f(0), 0.6599)
s2 = steady_state_model(72, c_to_f(0), 0.6599)
s3 = steady_state_model(76, c_to_f(0), 0.6599)

Mode: heating
Internal RH:  0.17242505971266292
Building heat loss:  709.1956521739131
Wall temperature (F):  65.62926024844717

Heat loss of user (watts):  69.4420056511378
Verdict:  Good

Mode: heating
Internal RH:  0.15042791978650996
Building heat loss:  787.9951690821259
Wall temperature (F):  69.36584472049695

Heat loss of user (watts):  59.86750303131723
Verdict:  Good

Mode: heating
Internal RH:  0.13154044151835576
Building heat loss:  866.7946859903386
Wall temperature (F):  73.10242919254664

Heat loss of user (watts):  50.153339249933424
Verdict:  Good



In [23]:
# February 2026, row 2 (2/1/2026  12:15:00 AM)
s = steady_state_model(68, c_to_f(-11.4), 0.6064)
s2 = steady_state_model(72, c_to_f(-11.4), 0.6064)
s3 = steady_state_model(76, c_to_f(-11.4), 0.6064)

Mode: heating
Internal RH:  0.0664313854131188
Building heat loss:  1113.4371739130427
Wall temperature (F):  64.2779385900621

Heat loss of user (watts):  72.03771095434932
Verdict:  Cold

Mode: heating
Internal RH:  0.057956396436135114
Building heat loss:  1192.2366908212557
Wall temperature (F):  68.01452306211179

Heat loss of user (watts):  62.440051730775735
Verdict:  Good

Mode: heating
Internal RH:  0.050679488135192186
Building heat loss:  1271.0362077294683
Wall temperature (F):  71.75110753416148

Heat loss of user (watts):  52.71292248096523
Verdict:  Good



In [34]:
# April 2022, row 1582 (4/25/2022  5:55:00 PM)
s = steady_state_model(68, c_to_f(27.9), 0.3289)
s2 = steady_state_model(72, c_to_f(27.9), 0.3289)
s3 = steady_state_model(76, c_to_f(27.9), 0.3289)

Mode: cooling
Internal RH:  0.5287823105309911
Building heat loss:  -280.1322826086949
Wall temperature (F):  68.9364422018634

Heat loss of user (watts):  62.5369200721063
Verdict:  Good

Mode: cooling
Internal RH:  0.46132286760193963
Building heat loss:  -201.33276570048207
Wall temperature (F):  72.67302667391309

Heat loss of user (watts):  53.079968213546096
Verdict:  Good

Mode: cooling
Internal RH:  0.40339993914025396
Building heat loss:  -122.53324879226929
Wall temperature (F):  76.40961114596277

Heat loss of user (watts):  43.44966434925645
Verdict:  Hot

